In [1]:
!nvidia-smi
!pip install -q transformers datasets accelerate scikit-learn

Tue Aug 25 17:46:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import files
uploaded = files.upload()

Saving test.jsonl to test.jsonl
Saving val.jsonl to val.jsonl
Saving train.jsonl to train.jsonl


In [3]:
!pip install -q git+https://github.com/Ikbola/uznorm.git

from uznorm.normalize import normalize_apostrophes
print([n for n in dir(normalize_apostrophes) if not n.startswith("_")])

PROBE = "O\u2018zbekiston va Farg\u2018ona \u2014 ta\u2019lim"
print("raw:", PROBE)

normalize = normalize_apostrophes
print("norm:", normalize(PROBE))

assert normalize(PROBE) != PROBE, (
    "uznorm did not change U+2018/U+2019 text. The ablation would be a no-op. "
    "Extend uznorm to map U+2018/U+2019 before continuing."
)
print("\nOK — normalization changes kun.uz-style apostrophes")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[]
raw: O‘zbekiston va Farg‘ona — ta’lim
norm: Oʻzbekiston va Fargʻona — taʼlim

OK — normalization changes kun.uz-style apostrophes


In [4]:
import numpy as np, pandas as pd, torch, random
from datasets import Dataset

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

splits = {s: pd.read_json(f"{s}.jsonl", lines=True) for s in ["train", "val", "test"]}
LABELS = sorted(splits["train"]["category"].unique())
L2I = {l: i for i, l in enumerate(LABELS)}
print(LABELS, {s: len(d) for s, d in splits.items()})

def build(df, do_norm):
    text = (df["title"].fillna("") + "\n\n" + df["text"]).tolist()
    if do_norm:
        text = [normalize(t) for t in text]
    return Dataset.from_dict({
        "text": text,
        "label": [L2I[c] for c in df["category"]],
    })

sample = (splits["train"]["title"].fillna("") + "\n\n" + splits["train"]["text"]).head(200)
changed = sum(normalize(t) != t for t in sample)
print(f"normalization changes {changed}/200 training documents")

['iqtisodiyot', 'jahon', 'jamiyat', 'sport', 'uzbekiston'] {'train': 2164, 'val': 270, 'test': 271}
normalization changes 200/200 training documents


In [5]:
import inspect
from transformers import TrainingArguments
import transformers

print("transformers", transformers.__version__)
SUPPORTED = set(inspect.signature(TrainingArguments.__init__).parameters)
for name in ["warmup_ratio", "eval_strategy", "evaluation_strategy",
             "save_strategy", "weight_decay", "logging_steps",
             "fp16", "report_to", "learning_rate", "seed"]:
    print(f"  {name:22} {'yes' if name in SUPPORTED else 'NO'}")

transformers 5.15.0
  warmup_ratio           NO
  eval_strategy          yes
  evaluation_strategy    NO
  save_strategy          yes
  weight_decay           yes
  logging_steps          yes
  fp16                   yes
  report_to              yes
  learning_rate          yes
  seed                   yes


In [6]:
import inspect
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

MODEL = "xlm-roberta-base"
MAX_LEN = 512

ARG_PARAMS = set(inspect.signature(TrainingArguments.__init__).parameters)
TRAINER_PARAMS = set(inspect.signature(Trainer.__init__).parameters)


def metrics(p):
    preds = p.predictions.argmax(-1)
    return {
        "accuracy": accuracy_score(p.label_ids, preds),
        "macro_f1": f1_score(p.label_ids, preds, average="macro"),
    }


def run(do_norm, tag):
    print(f"\n{'='*60}\n{tag}\n{'='*60}")
    tok = AutoTokenizer.from_pretrained(MODEL)

    def enc(b):
        return tok(b["text"], truncation=True, max_length=MAX_LEN, padding="max_length")

    ds = {s: build(df, do_norm).map(enc, batched=True) for s, df in splits.items()}

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL, num_labels=len(LABELS),
        id2label={i: l for l, i in L2I.items()}, label2id=L2I,
    )

    wanted = dict(
        output_dir=f"out_{tag}",
        num_train_epochs=4,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        seed=SEED,
        fp16=torch.cuda.is_available(),
        report_to="none",
    )
    kwargs = {k: v for k, v in wanted.items() if k in ARG_PARAMS}
    dropped = sorted(set(wanted) - set(kwargs))
    if dropped:
        print("dropped unsupported args:", dropped)

    args = TrainingArguments(**kwargs)

    tkw = dict(
        model=model, args=args,
        train_dataset=ds["train"], eval_dataset=ds["val"],
        compute_metrics=metrics,
    )
    if "processing_class" in TRAINER_PARAMS:
        tkw["processing_class"] = tok
    elif "tokenizer" in TRAINER_PARAMS:
        tkw["tokenizer"] = tok

    trainer = Trainer(**tkw)
    trainer.train()

    out = trainer.predict(ds["test"])
    preds = out.predictions.argmax(-1)
    print(classification_report(out.label_ids, preds, target_names=LABELS, digits=3))

    return {
        "tag": tag,
        "accuracy": float(accuracy_score(out.label_ids, preds)),
        "macro_f1": float(f1_score(out.label_ids, preds, average="macro")),
        "confusion_matrix": confusion_matrix(out.label_ids, preds).tolist(),
    }, trainer, tok

In [7]:
res_raw, trainer_raw, tok_raw = run(False, "raw")
res_norm, trainer_norm, tok_norm = run(True, "normalized")


raw


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


dropped unsupported args: ['evaluation_strategy', 'warmup_ratio']


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,0.735660,0.560136,0.814815,0.812107
2,0.531120,0.518765,0.800000,0.798869
3,0.434648,0.510764,0.822222,0.813882
4,0.344464,0.536254,0.800000,0.799195


              precision    recall  f1-score   support

 iqtisodiyot      0.672     0.759     0.713        54
       jahon      0.891     0.907     0.899        54
     jamiyat      0.729     0.648     0.686        54
       sport      0.964     0.982     0.973        55
  uzbekiston      0.549     0.519     0.533        54

    accuracy                          0.764       271
   macro avg      0.761     0.763     0.761       271
weighted avg      0.762     0.764     0.762       271



In [8]:
import json

print(f"{'variant':12} {'accuracy':>10} {'macro-F1':>10}")
print(f"{'baseline':12} {0.7860:>10.4f} {0.7814:>10.4f}")
for r in (res_raw, res_norm):
    print(f"{r['tag']:12} {r['accuracy']:>10.4f} {r['macro_f1']:>10.4f}")

delta = res_norm["accuracy"] - res_raw["accuracy"]
print(f"\nnormalization delta: {delta:+.4f} accuracy "
      f"({delta * len(splits['test']):+.1f} of {len(splits['test'])} test articles)")

with open("finetune_results.json", "w") as f:
    json.dump({"raw": res_raw, "normalized": res_norm, "labels": LABELS}, f, indent=2)

best = trainer_norm if res_norm["accuracy"] >= res_raw["accuracy"] else trainer_raw
best.save_model("model"); (tok_norm if best is trainer_norm else tok_raw).save_pretrained("model")
!zip -qr model.zip model

files.download("finetune_results.json")
files.download("model.zip")

variant        accuracy   macro-F1
baseline         0.7860     0.7814
raw              0.7638     0.7574
normalized       0.7638     0.7609

normalization delta: +0.0000 accuracy (+0.0 of 271 test articles)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>